In [1]:
import requests, feedparser, json,time, os, xmltodict
import polars as pl
from config_info import APIS
from pprint import pprint

from scrapers.arxiv import parser_arxiv
from scrapers.hal import parser_hal

# Basic Fetch

In [2]:
def fetch_raw(url, headers=None):
    headers = headers or {"User-Agent":  "IntelliCorpus/1.0 (contact: paull@scholar-cergy.com)"}
    r = requests.get(url, headers=headers, timeout=15)
    r.raise_for_status()
    content_type = r.headers.get("Content-Type","")
    if 'xml' in content_type:
        return r.text
    elif 'json' in content_type:
        return r.json()
    else:
        return r.text

# Semantic Scholar

In [3]:
def semantic_fetch(results):
    for result in results["data"]:
        print(result)
        # query = APIS["Semantic Scholar"]["paper_url"].format(paper_id=result["paperId"])
        # res = fetch_raw(query)
        # print(res)
        # print(query)

# CORE

In [4]:
# from dotenv import load_dotenv
# load_dotenv()
# API_KEY = os.getenv("CORE_API_KEY")
# url = "https://api.core.ac.uk/v3/search/works"

# headers = {
#     "Authorization": f"Bearer {API_KEY}"
# }

# params = { 
#     "q": "AI agent",
#     "limit": 5
# }

# r = requests.get(url, headers=headers, params=params)
# r.raise_for_status()

# data = r.json()
# # print(data.keys())
# pprint(data)


# Main 

In [5]:
# raw_result = fetch_raw(APIS["arXiv"]["api_url"].format(query="AI agent",quantity='1'))
# result_arxiv =  parser_arxiv(raw_result)
# print(type(result_arxiv)) # Arxiv Ok
 
# Test HAL
# url_hal = fetch_raw(APIS["HAL"]["api_url"].format(query="AI agent",quantity='2'))
# result_hal = parser_hal(url_hal)
# pprint(result_hal[0]) #Good 

# Test PubMed
from scrapers.pubmed import fetch_pubmed
xml_batches_pubmed = fetch_pubmed(
    query="AI agent",
    max_results=1,
    batch_size=1,
    email="proliquer@scholar-perigueuxu.com"
)
pprint(xml_batches_pubmed)

# Test Sementic Scholar
# url_sem = fetch_raw(APIS["Semantic Scholar"]["api_url"].format(query="AI agent",quantity='5'))
# print(type(url_sem))
# semantic_fetch(url_sem)


['<?xml version="1.0" ?>\n'
 '<!DOCTYPE PubmedArticleSet PUBLIC "-//NLM//DTD PubMedArticle, 1st January '
 '2025//EN" "https://dtd.nlm.nih.gov/ncbi/pubmed/out/pubmed_250101.dtd">\n'
 '<PubmedArticleSet>\n'
 '<PubmedArticle><MedlineCitation Status="Publisher" Owner="NLM"><PMID '
 'Version="1">41862602</PMID><DateRevised><Year>2026</Year><Month>03</Month><Day>21</Day></DateRevised><Article '
 'PubModel="Print-Electronic"><Journal><ISSN '
 'IssnType="Electronic">1546-1696</ISSN><JournalIssue '
 'CitedMedium="Internet"><PubDate><Year>2026</Year><Month>Mar</Month><Day>20</Day></PubDate></JournalIssue><Title>Nature '
 'biotechnology</Title><ISOAbbreviation>Nat '
 'Biotechnol</ISOAbbreviation></Journal><ArticleTitle>Generalist biological '
 'artificial intelligence in modeling the language of '
 'life.</ArticleTitle><ELocationID EIdType="doi" '
 'ValidYN="Y">10.1038/s41587-026-03064-w</ELocationID><Abstract><AbstractText>Generalist '
 'biological artificial intelligence (GBAI) represents a tr

In [6]:
from scrapers.pubmed import format_hal_data
pubmed_list = format_hal_data(xml_batches_pubmed)


In [ ]:
from database.postgres.crud import upsert_data
from models.postgres.corpus_schema import document_table 
from config.db_engine import get_db_engine
from processing.cleaning_data import normalize_data

arxiv_mapping = {
    "published_at": "published",
}
# raw_result = fetch_raw(APIS["arXiv"]["api_url"].format(query="AI agent",quantity='1'))
# result_arxiv =  parser_arxiv(raw_result)
# pprint(result_arxiv)
# clean_arxiv_data = normalize_data(
#     raw_data=result_arxiv, 
#     source_name="arXiv",
#     date_columns=["published_at"],
#     columns_drop=["updated"]
#     )


# hal_mapping = { "published" : "published_at"}
# clean_hal_data = normalize_data(
#     raw_data=result_hal,
#     source_name="Hal",
#     column_mapping=hal_mapping,
#     date_columns=["published"]
# )

clean_pubmed_data = normalize_data(
    raw_data= pubmed_list,
    source_name="Pubmed",
    date_columns=["published"]
)
print(clean_pubmed_data)
engine = get_db_engine()
upsert_data(clean_pubmed_data, ['id'], document_table, engine)


[{'id': '1cb8453c-259d-5b4c-a103-4a01b9f12d14', 'title': 'Generalist biological artificial intelligence in modeling the language of life.', 'summary': "Generalist biological artificial intelligence (GBAI) represents a transformative approach to modeling the 'language of life'-the flow of information from DNA to cellular function. This Review synthesizes rapid advances in biological AI to interpret and generate DNA, RNA, proteins and cellular systems. We chart a course toward comprehensive systems that can concurrently process and predict across these domains, performing several critical biological tasks simultaneously. Substantial opportunities lie in synergizing language and structural AI, leveraging specialized models and improving AI agents for autonomous discovery. After addressing challenges in data, biological complexity, scalability and experimental validation, GBAI has the potential to deepen our understanding of disease pathways and biomarkers, advance automated therapeutic de

: 